In [2]:
import pandas as pd
import numpy as np
pd.options.display.max_rows = 999

In [3]:
class BuySell:

    def __init__(self, capital):
        self.initial_capital = capital
        self.current_capital = capital
        self.share_amount = 0


    def buyandhold(self, dataframe):
        """

        Args:
            dataframe: (pd.DataFrame) should contains price and directive, date columns

        Returns: (float) profit

        """
        first_price = dataframe.iloc[0].loc['price']
        last_price  = dataframe.iloc[-1].loc['price']

        self.current_capital, self.share_amount = BuySell._buy(self.initial_capital, first_price)
        money, self.share_amount = BuySell._sell(self.share_amount, last_price)
        self.current_capital += money

        profit = self.current_capital - self.initial_capital

        return profit

    @staticmethod
    def _buy(money, price):
        """buys if appliable
        Returns: (float, int) current_money, share_amount"""

        remaining_money = money
        share_amount = 0

        if money > price:
            share_amount = int(money / price)
            remaining_money = money - share_amount * price

        return remaining_money, share_amount

    @staticmethod
    def _sell(share_amount, price):
        """sells all shares :)
        Returns: (float, int) current_money, share_amount"""
        money = share_amount * price
        return money, 0



    def process(self, dataframe):
        """

        Args:
            dataframe: (pd.DataFrame) should contains price and directive, date columns

        Returns: (float) profit

        """
        capital_list = list()
        share_amount_list = list()

        for idx,row in dataframe.iterrows():
            if row['directive'] == 'buy':
                remaining_money, share_amount = BuySell._buy(self.current_capital, row['price'])
                self.current_capital = remaining_money
                self.share_amount += share_amount

            if row['directive'] == 'sell':
                money, share_amount = BuySell._sell(self.share_amount, row['price'])
                self.current_capital += money
                self.share_amount = share_amount
            capital_list.append(self.current_capital)
            share_amount_list.append(self.share_amount)

        dataframe.loc[:,'current_capital'] = capital_list
        dataframe.loc[:,'share_amount'] = share_amount_list
        dataframe.loc[:,'total_capital'] = dataframe['current_capital'] + dataframe['share_amount']*dataframe['price']

        dataframe.loc[:,'profit'] = dataframe['total_capital'] - self.initial_capital


        return dataframe


In [4]:
def label_to_directives(row):
    row = row[['pbuy','psell','phold']]
    argmax_idx = np.argmax(row.values)

    if argmax_idx == 0:
        return 'buy'
    if argmax_idx == 1:
        return 'sell'
    return 'hold'


In [10]:
# np.random.seed(42)
# size = 100
# fake_dataframe = pd.DataFrame({'price':np.random.randint(100,150, size=size),
#                                'directive':np.random.choice(['buy','hold','sell'], size=size, replace=True),
#      'name':np.random.choice(['xlp','xlu','xlv','xly','dia','ewa','ewc','ewg','ewh','ewj','eww','spy','xlb','xle','xlf','xli','xlk'], size=size, replace=True)})


In [33]:
initial_capital = 100000
# stock_names = ['xlp','xlu','xlv','xly','dia','ewa','ewc','ewg','ewh','ewj','eww','spy','xlb','xle','xlf','xli','xlk']
exp_name = 'exp2'
stock_names = ['dia','ewa']
final_result_df = None
for stock_name in stock_names:
    dataframe = pd.read_csv('../experiment/finance_cnn_{}_{}_only/prediction_results.csv'.format(exp_name,stock_name))
    dataframe['directive'] = dataframe.apply(label_to_directives, axis=1)
    dataframe['price'] = dataframe['raw_adjusted_close'].values
    
    # Buy and Hold strategy
    buyandhold_profit = BuySell(capital=initial_capital).buyandhold(dataframe)
    
    # Our simple strategy
    buysell = BuySell(capital=initial_capital)
    buysell_result_df = buysell.process(dataframe)
    subset = ['name','date','price','directive','current_capital','share_amount','total_capital', 'profit']
    if final_result_df is None:
        final_result_df = pd.DataFrame(columns=subset)
    
    last_row = buysell_result_df[subset].iloc[-1]
    last_row['bah_profit'] = buyandhold_profit
    
    final_result_df = final_result_df.append(last_row)
    
final_result_df = final_result_df.reset_index(drop=True)



In [34]:
final_result_df

,name,date,price,directive,current_capital,share_amount,total_capital,profit,bah_profit
0,dia,2016-10-18,181.37,hold,147.30,660,119851.5,19851.5,3637.27
1,ewa,2016-10-31,20.28,hold,1.66,5943,120525.7,20525.7,-7184.32
